# 3D Porous Media — Two Spheres (LBM Simulation)
## JAX D3Q19 Lattice Boltzmann Method with Zou-He Boundary Conditions

This notebook runs a 3D Lattice Boltzmann simulation of flow through a porous medium
consisting of two spheres, using the D3Q19 lattice model implemented in JAX.

**Features:**
- D3Q19 lattice with BGK collision operator
- Zou-He pressure boundary conditions at inlet/outlet
- Bounce-back for solid nodes
- Convergence check based on mean velocity
- Permeability calculation and comparison with Palabos reference

## 1. Imports

In [ ]:
import jax
import jax.numpy as jnp
from jax import jit
import numpy as np
import time

print(f"JAX devices: {jax.devices()}")

## 2. Simulation Parameters

These parameters are set to match the Palabos reference case.

In [ ]:
# Grid dimensions
NX, NY, NZ = 48, 64, 64

# Relaxation parameters
OMEGA = 1.0
TAU = 1.0
NU = 1.0 / 6.0

# Pressure drop
DELTA_P = 0.00005
RHO_IN = 1.0
RHO_OUT = 1.0 - DELTA_P * 3.0

# Simulation control
MAX_STEPS = 30000
PRINT_EVERY = 500
CONV = 1e-6

# Lattice directions
Q = 19

print(f"Grid: {NX}x{NY}x{NZ}  omega={OMEGA}  nu={NU:.6f}  dP={DELTA_P}")

## 3. D3Q19 Lattice Definition

Define the lattice velocities, weights, and opposite directions for the D3Q19 model.

In [ ]:
# D3Q19 lattice velocities
C = np.array([
    [0,0,0],                                          # rest
    [1,0,0],[-1,0,0],[0,1,0],[0,-1,0],[0,0,1],[0,0,-1],  # face neighbours
    [1,1,0],[-1,-1,0],[1,-1,0],[-1,1,0],              # edge neighbours (xy)
    [1,0,1],[-1,0,-1],[1,0,-1],[-1,0,1],              # edge neighbours (xz)
    [0,1,1],[0,-1,-1],[0,1,-1],[0,-1,1]               # edge neighbours (yz)
], dtype=np.int32)

# Weights
W = np.array([1./3.] + [1./18.]*6 + [1./36.]*12)

# Opposite direction indices
OPP = np.array([0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15, 18, 17])

# Reshaped arrays for broadcasting (4D: Q x NX x NY x NZ)
cx4 = jnp.array(C[:,0]).reshape(Q,1,1,1)
cy4 = jnp.array(C[:,1]).reshape(Q,1,1,1)
cz4 = jnp.array(C[:,2]).reshape(Q,1,1,1)
w4  = jnp.array(W).reshape(Q,1,1,1)

# Reshaped arrays for broadcasting (3D: Q x NY x NZ) — used in BC
cx2 = jnp.array(C[:,0]).reshape(Q,1,1)
cy2 = jnp.array(C[:,1]).reshape(Q,1,1)
cz2 = jnp.array(C[:,2]).reshape(Q,1,1)
w2  = jnp.array(W).reshape(Q,1,1)
opp = jnp.array(OPP)

# Direction groups for Zou-He BC
POS_X  = jnp.array([1, 7, 9, 11, 13])   # positive x-velocity directions
NEG_X  = jnp.array([2, 8, 10, 12, 14])  # negative x-velocity directions
ZERO_X = jnp.array([0, 3, 4, 5, 6, 15, 16, 17, 18])  # zero x-velocity

print("D3Q19 lattice initialised.")

## 4. Geometry Loading

Read the `twoSpheres.dat` geometry file. Node types:
- **0** = fluid
- **1** = bounce-back (solid)
- **2** = no-dynamics (exterior)

In [ ]:
print("Reading geometry...")
geo = np.loadtxt("twoSpheres.dat", dtype=int).reshape(NX, NY, NZ)
geo[-1, :, :] = 0  # Palabos reads only nx-1 slices

# Boolean masks (with leading singleton dim for broadcasting with f)
is_fluid = jnp.array(geo == 0)[None]
is_bb    = jnp.array(geo == 1)[None]
is_nd    = jnp.array(geo == 2)[None]

# Inlet / outlet masks — interior fluid cells on x-faces
inm  = np.zeros((NY, NZ), bool)
outm = np.zeros((NY, NZ), bool)
inm[1:NY-1, 1:NZ-1]  = (geo[0,  1:NY-1, 1:NZ-1] == 0)
outm[1:NY-1, 1:NZ-1] = (geo[-1, 1:NY-1, 1:NZ-1] == 0)
inlet_mask  = jnp.array(inm)[None]
outlet_mask = jnp.array(outm)[None]

print(f"  Fluid = {int(np.sum(geo==0))}")
print(f"  BB    = {int(np.sum(geo==1))}")
print(f"  ND    = {int(np.sum(geo==2))}")

## 5. Core LBM Functions

Equilibrium distribution, macroscopic quantities, streaming, and boundary conditions.

In [ ]:
def eq3d(rho, ux, uy, uz):
    """Equilibrium distribution (full 3D field)."""
    usq = ux*ux + uy*uy + uz*uz
    cu  = cx4*ux[None] + cy4*uy[None] + cz4*uz[None]
    return w4 * rho[None] * (1 + 3*cu + 4.5*cu*cu - 1.5*usq[None])

def eq2d(rho, ux, uy, uz):
    """Equilibrium distribution (2D slice, for BCs)."""
    usq = ux*ux + uy*uy + uz*uz
    cu  = cx2*ux[None] + cy2*uy[None] + cz2*uz[None]
    return w2 * rho[None] * (1 + 3*cu + 4.5*cu*cu - 1.5*usq[None])

def macro(f):
    """Compute macroscopic quantities from distribution."""
    rho = jnp.sum(f, 0)
    ux  = jnp.sum(f * cx4, 0) / rho
    uy  = jnp.sum(f * cy4, 0) / rho
    uz  = jnp.sum(f * cz4, 0) / rho
    return rho, ux, uy, uz

def stream(f):
    """Streaming step: shift each population along its lattice velocity."""
    out = jnp.zeros_like(f)
    for q in range(Q):
        s = jnp.roll(f[q], C[q,0], 0)
        s = jnp.roll(s,    C[q,1], 1)
        s = jnp.roll(s,    C[q,2], 2)
        out = out.at[q].set(s)
    return out

print("Core functions defined.")

## 6. Zou-He Pressure Boundary Conditions

Pressure-driven flow: set density (pressure) at inlet and outlet faces.

In [ ]:
def zou_he_inlet(f):
    """Apply Zou-He pressure BC at inlet (x = 0)."""
    s = f[:, 0, :, :]
    rho_in = RHO_IN * jnp.ones((NY, NZ))
    ux_in  = 1.0 - (jnp.sum(s[ZERO_X], 0) + 2*jnp.sum(s[NEG_X], 0)) / rho_in
    feq    = eq2d(rho_in, ux_in, jnp.zeros((NY,NZ)), jnp.zeros((NY,NZ)))
    return f.at[:, 0, :, :].set(jnp.where(inlet_mask, feq, s))

def zou_he_outlet(f):
    """Apply Zou-He pressure BC at outlet (x = NX-1)."""
    s = f[:, -1, :, :]
    rho_out = RHO_OUT * jnp.ones((NY, NZ))
    ux_out  = -1.0 + (jnp.sum(s[ZERO_X], 0) + 2*jnp.sum(s[POS_X], 0)) / rho_out
    feq     = eq2d(rho_out, ux_out, jnp.zeros((NY,NZ)), jnp.zeros((NY,NZ)))
    return f.at[:, -1, :, :].set(jnp.where(outlet_mask, feq, s))

print("Boundary condition functions defined.")

## 7. LBM Time Step

One complete LBM step: collision → streaming → boundary conditions.

In [ ]:
@jit
def one_step(f, is_fluid, is_bb, is_nd):
    """Single LBM time step."""
    # Collision: fluid → BGK, bounce-back → swap, no-dynamics → skip
    _, ux, uy, uz = macro(f)
    rho = jnp.sum(f, 0)
    feq = eq3d(rho, ux, uy, uz)
    f_coll = jnp.where(is_bb, f[opp],
             jnp.where(is_nd, f,
                        f - (f - feq) / TAU))
    # Streaming
    f_str = stream(f_coll)
    # Zou-He pressure BC
    f_str = zou_he_inlet(f_str)
    f_str = zou_he_outlet(f_str)
    return f_str

print("one_step function defined.")

## 8. Initialisation

Start with a linear pressure gradient from inlet to outlet.

In [ ]:
print("Initialising distribution functions...")
x = np.arange(NX, dtype=np.float64)
rho0 = jnp.array(
    np.broadcast_to(
        (1.0 - DELTA_P * 3 / (NX-1) * x)[:, None, None],
        (NX, NY, NZ)
    ).copy()
)
z = jnp.zeros((NX, NY, NZ))
f = eq3d(rho0, z, z, z)

# JIT warm-up
print("JIT compiling (first call)...")
t0 = time.time()
f = one_step(f, is_fluid, is_bb, is_nd)
f.block_until_ready()
print(f"JIT compilation took {time.time()-t0:.1f} s")

## 9. Main Simulation Loop

Run until convergence or `MAX_STEPS` is reached.

In [ ]:
print(f"Running simulation (max {MAX_STEPS} steps)...")
t0 = time.time()
prev = 0.0

for step in range(1, MAX_STEPS + 1):
    f = one_step(f, is_fluid, is_bb, is_nd)

    if step % PRINT_EVERY == 0:
        f.block_until_ready()
        rho, ux, uy, uz = macro(f)
        mu  = float(jnp.sum(ux) / (NX * NY * NZ))
        rel = abs(mu - prev) / abs(mu) if mu != 0 else 1.0
        print(f"  step {step:5d}  <u_x> = {mu:.6e}  rel_change = {rel:.2e}")
        if rel < CONV and step > 1000:
            print("  >>> CONVERGED")
            break
        prev = mu

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f} s  ({NX*NY*NZ*step/elapsed/1e6:.0f} MLUPS)")

## 10. Results & Permeability

Compute the permeability using Darcy's law and compare with the Palabos reference value.

In [ ]:
# Final macroscopic fields
rho, ux, uy, uz = macro(f)
mu = float(jnp.sum(ux) / (NX * NY * NZ))
K  = NU * mu / (DELTA_P / (NX - 1))

print("=" * 50)
print(f"  Permeability   = {K:.4f}")
print(f"  Palabos ref    = 27.76")
print(f"  Error          = {abs(K - 27.76) / 27.76 * 100:.1f}%")
print("=" * 50)

## 11. Save Results

In [ ]:
np.savez("lbm_twospheres_results.npz",
    geometry=geo,
    ux=np.array(ux), uy=np.array(uy), uz=np.array(uz), rho=np.array(rho),
    permeability=K, mean_ux=mu,
    NX=NX, NY=NY, NZ=NZ, nu=NU, deltaP=DELTA_P
)
print("Saved → lbm_twospheres_results.npz")